# Python 1 天 8 小时最小必要集 · 交互验证版

面向 10 年 Java 经验、Python 零基础。本 notebook 是 [完整课程](course.zh.md) 的可运行孪生体：**从上到下依次执行即可复现文中全部 ✅ 结果**，也可以随时改参数实时验证。

- 无需 API Key：LLM/embedding 走 mock 双实现；设置 `LLM_API_KEY` 环境变量后自动切真实厂商。
- 依赖：`numpy pandas httpx openai fastapi pytest`（Colab 已内置大部分，缺的 cell 会提示）。
- 本地运行：`uv run --with jupyterlab jupyter lab course.ipynb`

## P1 · 环境与语法差异

Java 类比：无编译步骤、缩进即语法、`==` 比值 / `is` 比引用（与 Java 正好相反）、`None` ≈ null。环境用 uv（≈ Maven），命令见课程文档；这里直接跑语法。

In [1]:
model = "qwen-plus"
messages = [
    {"role": "system", "content": "你是资深Java导师"},
    {"role": "user", "content": "Python的dict相当于Java的什么?"},
]
print(f"model={model}, turns={len(messages)}")
for i, msg in enumerate(messages):
    print(f"[{i}] {msg['role']}: {msg['content'][:20]}")

a = [{"role": "user"}]
b = [{"role": "user"}]
print("== 比值:", a == b, "| is 比引用:", a is b)

reply = None
print("判空固定写法:", reply is None)
print("5/2 =", 5 / 2, "| 5//2 =", 5 // 2)

model=qwen-plus, turns=2
[0] system: 你是资深Java导师
[1] user: Python的dict相当于Java的什
== 比值: True | is 比引用: False
判空固定写法: True
5/2 = 2.5 | 5//2 = 2


In [2]:
def estimate_tokens(text: str) -> int:
    return len(text) // 2

def estimate_cost(tokens: int, price_per_1k: float) -> float:
    return tokens / 1000 * price_per_1k

print(estimate_tokens("你好世界"))
print(f"cost={estimate_cost(2000, 0.8):.4f}元")

2
cost=1.6000元


## P2 · 类型标注 + 容器 + 推导式

标注运行时不强制（≈ 写给 IDE 的合同）；推导式 ≈ 压缩版 Stream API。动手：改下面的过滤条件，实时看结果。

In [3]:
def token_stats(msgs: list[dict[str, str]]) -> tuple[int, float]:
    counts = [len(m["content"]) // 2 for m in msgs]
    return sum(counts), sum(counts) / len(counts)

messages = [
    {"role": "system", "content": "你是Java转AI导师"},
    {"role": "user", "content": "解释一下RAG"},
    {"role": "assistant", "content": "RAG=检索+生成"},
    {"role": "user", "content": "给个代码例子"},
]
print("token_stats:", token_stats(messages))          # (15, 3.75)
print("filter+map:", [m["content"] for m in messages if m["role"] == "user"])
print("anyMatch:", any(m["role"] == "tool" for m in messages))

scored = [{"text": "RAG定义", "score": 0.9}, {"text": "无关", "score": 0.2}, {"text": "RAG流程", "score": 0.8}]
print("top-k:", [c["text"] for c in sorted(scored, key=lambda c: c["score"], reverse=True)[:2]])

token_stats: (15, 3.75)
filter+map: ['解释一下RAG', '给个代码例子']
anyMatch: False
top-k: ['RAG定义', 'RAG流程']


In [4]:
# 坑现场：引用 vs 拷贝；[[]]*3 共享
a = [1, 2]; b = a; c = a[:]
b.append(3)
print("a:", a, "c:", c)

grid = [[]] * 3
grid[0].append(1)
print("[[]]*3 陷阱:", grid)
print("正确写法:", [[x] for x in range(3)])

a: [1, 2, 3] c: [1, 2]
[[]]*3 陷阱: [[1], [1], [1]]
正确写法: [[0], [1], [2]]


## P3 · 函数/装饰器 + dataclass

装饰器 = 主动函数包装（`@retry` ≡ `f = retry(f)`），Java 注解是 passive 元数据。dataclass ≈ Lombok `@Data`。

In [5]:
def build_messages(prompt: str, system: str = "你是AI助手", history: list | None = None) -> list[dict]:
    if history is None:
        history = []
    return [{"role": "system", "content": system}, *history, {"role": "user", "content": prompt}]

def make_params(prompt: str, **options) -> dict:
    return {"model": "qwen-plus", "messages": build_messages(prompt), **options}

print(make_params("解释RAG", temperature=0.1))

{'model': 'qwen-plus', 'messages': [{'role': 'system', 'content': '你是AI助手'}, {'role': 'user', 'content': '解释RAG'}], 'temperature': 0.1}


In [6]:
# 头号坑：可变默认参数跨调用共享
def bad_append(msg, history=[]):
    history.append(msg)
    return history
print(bad_append("a"))
print(bad_append("b"))   # 残留！修复：默认 None

['a']
['a', 'b']


In [7]:
import time
from functools import wraps

def retry(fn):
    @wraps(fn)
    def wrapper(*args, **kwargs):
        for attempt in range(3):
            try:
                return fn(*args, **kwargs)
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(0.1 * (attempt + 1))
    return wrapper

calls = {"n": 0}
@retry
def fake_llm(prompt: str) -> str:
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("限流")
    return f"回答:{prompt}"

print(fake_llm("什么是RAG"), "| 尝试:", calls["n"], "| 函数名:", fake_llm.__name__)

回答:什么是RAG | 尝试: 3 | 函数名: fake_llm


In [8]:
from dataclasses import dataclass, field, asdict

@dataclass
class ChatMessage:
    role: str
    content: str
    tokens: int = 0

m = ChatMessage("user", "解释RAG")
print(m)                                   # 自动 __repr__
print(m == ChatMessage("user", "解释RAG"))  # 自动 __eq__
print(asdict(m))                           # → dict，接 JSON

ChatMessage(role='user', content='解释RAG', tokens=0)
True
{'role': 'user', 'content': '解释RAG', 'tokens': 0}


## P4 · 文件/JSON + httpx + async

`with open` ≈ try-with-resources；json ≈ Jackson（记得 `ensure_ascii=False`）；asyncio = 单线程事件循环 ≈ Netty IO 线程，`await` 处让出控制权。

In [9]:
import json, tempfile
from pathlib import Path

p = Path(tempfile.mkdtemp()) / "history.json"
msgs = [ChatMessage("user", "解释RAG"), ChatMessage("assistant", "RAG=检索+生成")]
p.write_text(json.dumps([asdict(m) for m in msgs], ensure_ascii=False, indent=2), encoding="utf-8")
loaded = [ChatMessage(**d) for d in json.loads(p.read_text(encoding="utf-8"))]
print("roundtrip:", loaded == msgs)
print("默认转义:", json.dumps({"q": "你好"}))
print("正确:", json.dumps({"q": "你好"}, ensure_ascii=False))

roundtrip: True
默认转义: {"q": "\u4f60\u597d"}
正确: {"q": "你好"}


In [10]:
import httpx
try:
    r = httpx.get("https://httpbin.org/get", params={"q": "rag"}, timeout=10)
    r.raise_for_status()
    print("GET args:", r.json()["args"])
    r = httpx.get("https://wttr.in/Beijing?format=j1", timeout=10)
    print("北京温度:", r.json()["current_condition"][0]["temp_C"])
except Exception as e:
    print("网络不可达，跳过:", type(e).__name__)

GET args: {'q': 'rag'}


北京温度: 9


In [11]:
import asyncio, time

async def fake_embed(text: str) -> list[float]:
    await asyncio.sleep(0.3)
    return [len(text) * 0.1]

async def bad_embed(text: str) -> list[float]:
    time.sleep(0.3)   # 阻塞事件循环！
    return [len(text) * 0.1]

t0 = time.perf_counter()
print("await 并发:", await asyncio.gather(fake_embed("你好"), fake_embed("RAG")), f"{time.perf_counter()-t0:.2f}s")
t0 = time.perf_counter()
await asyncio.gather(bad_embed("你好"), bad_embed("RAG"))
print("time.sleep 阻塞:", f"{time.perf_counter()-t0:.2f}s")

await 并发: [[0.2], [0.30000000000000004]] 0.30s


time.sleep 阻塞: 0.61s


## P5 · NumPy（RAG 数学地基）

ndarray ≈ double[] + 整数组运算符。`*` 逐元素、`@` 点积；切片是**视图**不是拷贝。

In [12]:
import numpy as np

a = np.array([1.0, 2.0, 3.0]); b = np.array([4.0, 5.0, 6.0])
print("a+b:", a + b, "| a*b:", a * b, "| a@b:", a @ b)
print("argmax:", a.argmax())

def cosine(x, y):
    return float(x @ y / (np.linalg.norm(x) * np.linalg.norm(y)))

query = np.array([1.0, 0.0, 1.0])
docs = np.array([[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [0.9, 0.1, 0.8]])
sims = docs @ query / (np.linalg.norm(docs, axis=1) * np.linalg.norm(query))
print("sims:", sims, "| top2:", np.argsort(sims)[::-1][:2])

a+b: [5. 7. 9.] | a*b: [ 4. 10. 18.] | a@b: 32.0
argmax: 2
sims: [1.         0.         0.99484975] | top2: [0 2]


In [13]:
arr = np.array([1, 2, 3, 4])
view = arr[1:3]; view[0] = 99
print("视图改原数组:", arr)
xi = np.zeros(3, dtype=int); xi[0] = 0.9
print("int 截断:", xi)
print("广播扩维:", (np.array([1,2,3]) * np.array([[1],[2],[3]])).shape)

视图改原数组: [ 1 99  3  4]
int 截断: [0 0 0]
广播扩维: (3, 3)


## P6 · LLM API + embedding + 最小 RAG

OpenAI SDK ≈ JDBC：换厂商改 base_url/model/key。mock embedding 是字符哈希，**只演练管道形状，不代表语义**。

In [14]:
import os

def mock_embed(text: str, dim: int = 64) -> np.ndarray:
    v = np.zeros(dim)
    for ch in text:
        v[ord(ch) % dim] += 1.0
    return v

class FakeLLM:
    def chat(self, messages: list[dict]) -> str:
        return f"[mock] 已收到 {len(messages)} 条消息，最后一条：{messages[-1]['content'][:40]}..."

class OpenAILLM:
    def __init__(self):
        from openai import OpenAI
        self.client = OpenAI(api_key=os.environ["LLM_API_KEY"],
                             base_url=os.environ.get("LLM_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"))
        self.model = os.environ.get("LLM_MODEL", "qwen-plus")
    def chat(self, messages: list[dict]) -> str:
        return self.client.chat.completions.create(model=self.model, messages=messages).choices[0].message.content

def get_llm():
    if os.environ.get("LLM_API_KEY"):
        return OpenAILLM()
    print("⚠️ 未设置 LLM_API_KEY，使用 FakeLLM")
    return FakeLLM()

In [15]:
KB = [
    "RAG（检索增强生成）先从知识库检索相关片段，再交给大模型生成回答，能缓解幻觉并注入私有知识。",
    "embedding 把文本映射为定长浮点向量，语义相近的文本向量方向相近，常用余弦相似度衡量。",
    "FastAPI 是 Python 的异步 Web 框架，配合 pydantic 做请求校验，自带 /docs 交互式文档。",
    "今天北京晴，气温 8 摄氏度，适合出门散步。",
]

def build_index(chunks, embed):
    return np.array([embed(c) for c in chunks])

def retrieve(query, chunks, index, embed, k=2):
    q = embed(query)
    sims = index @ q / (np.linalg.norm(index, axis=1) * np.linalg.norm(q) + 1e-10)
    top = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in top]

def rag_answer(question, llm, embed, chunks, index):
    hits = retrieve(question, chunks, index, embed, k=2)
    context = "\n".join(f"[{i+1}] {c}" for i, (c, _) in enumerate(hits))
    messages = [
        {"role": "system", "content": "只根据提供的资料回答；资料不足就说不知道。"},
        {"role": "user", "content": f"资料：\n{context}\n\n问题：{question}"},
    ]
    return llm.chat(messages), hits

llm = get_llm()
index = build_index(KB, mock_embed)
for q in ["什么是RAG", "怎么衡量语义相似度"]:
    answer, hits = rag_answer(q, llm, mock_embed, KB, index)
    print(f"\nQ: {q} | top-1: {hits[0][0][:20]}... ({hits[0][1]:.3f})")

⚠️ 未设置 LLM_API_KEY，使用 FakeLLM

Q: 什么是RAG | top-1: RAG（检索增强生成）先从知识库检索相关... (0.408)

Q: 怎么衡量语义相似度 | top-1: embedding 把文本映射为定长浮点... (0.451)


## P7 · FastAPI + pytest

FastAPI ≈ Spring Boot，pydantic = DTO + Bean Validation + Jackson 三合一，校验失败自动 422。下面用 TestClient（≈ MockMvc）在 notebook 里直接打接口，无需起服务。

In [16]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

chat_app = FastAPI()

class ChatRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

class ChatResponse(BaseModel):
    answer: str
    model: str

@chat_app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest) -> ChatResponse:
    return ChatResponse(answer=f"[mock] 收到问题：{req.question[:40]}", model="mock")

client = TestClient(chat_app)
print("正常:", client.post("/chat", json={"question": "什么是RAG"}).json())
print("空 question:", client.post("/chat", json={"question": ""}).status_code)
print("temperature 越界:", client.post("/chat", json={"question": "hi", "temperature": 9}).status_code)

正常: {'answer': '[mock] 收到问题：什么是RAG', 'model': 'mock'}
空 question: 422


temperature 越界: 422


## P8 · 综合项目：/ask 返回 answer + sources

复用 P6 的检索 + P7 的 FastAPI。同步 LLM 调用放 `def` 端点（FastAPI 丢线程池），别放 async def 阻塞事件循环。

In [17]:
app = FastAPI(title="Java→AI 最小 RAG 服务")

class AskRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)
    top_k: int = Field(default=2, ge=1, le=5)

class Source(BaseModel):
    text: str
    score: float

class AskResponse(BaseModel):
    answer: str
    sources: list[Source]

@app.post("/ask", response_model=AskResponse)
def ask(req: AskRequest) -> AskResponse:
    hits = retrieve(req.question, KB, index, mock_embed, k=req.top_k)
    context = "\n".join(f"[{i+1}] {c}" for i, (c, _) in enumerate(hits))
    messages = [
        {"role": "system", "content": "只根据提供的资料回答；资料不足就说不知道。"},
        {"role": "user", "content": f"资料：\n{context}\n\n问题：{req.question}"},
    ]
    return AskResponse(answer=llm.chat(messages), sources=[Source(text=c, score=s) for c, s in hits])

c2 = TestClient(app)
r = c2.post("/ask", json={"question": "什么是RAG"})
body = r.json()
assert r.status_code == 200 and len(body["sources"]) == 2 and "RAG" in body["sources"][0]["text"]
print("status:", r.status_code)
print("sources:", [(s["text"][:18], round(s["score"], 3)) for s in body["sources"]])
print("校验:", c2.post("/ask", json={"question": ""}).status_code, c2.post("/ask", json={"question": "hi", "top_k": 99}).status_code)

status: 200
sources: [('RAG（检索增强生成）先从知识库检索', 0.408), ('FastAPI 是 Python 的', 0.235)]


校验: 422 422


## 下一步

- 改 `KB` 加一条你自己的知识，重跑 P6/P8 cell，看检索是否命中。
- 设 `LLM_API_KEY` 后重启 kernel，`get_llm()` 自动切真实厂商。
- 完整讲解、34 条坑汇总、29 项自测清单见 [course.zh.md](course.zh.md)；配套工程代码在 `code/`（含 CI）。